In [ ]:
import os
from langchain_tavily import TavilySearch
from dotenv import load_dotenv
load_dotenv()
from utils.logger import logger
from tools.search import web_search

# web_search = TavilySearch(max_results=3)
# web_search_results = web_search.invoke("who is the mayor of NYC?")

# print(web_search_results["results"][0]["content"])

In [ ]:
LLM = "ollama:llama3.2:latest"

# Legal Agent

In [ ]:
import sys
sys.path.append("../src")

from langgraph.prebuilt import create_react_agent
from tools.think import think_tool
from prompts.agents import LEGAL_AGENT_SYSTEM_MESSAGE
from utils.time import get_today_str
from pretify import pretty_print_messages

In [ ]:
legal_prompt = LEGAL_AGENT_SYSTEM_MESSAGE.format(date=get_today_str())

legal_agent = create_react_agent(
    model=LLM,
    tools=[web_search, think_tool],
    prompt=legal_prompt,
    name="legal_agent"
)

In [ ]:
for chunk in legal_agent.stream(
    {"messages": [{"role": "user", "content": "MarathonXP company from Kenya"}]}
):
    pretty_print_messages(chunk)
    logger.info(chunk)
    

# Finance Agent

In [ ]:
import sys
sys.path.append("../src")

from prompts.agents import  FINANCE_AGENT_SYSTEM_MESSAGE

In [ ]:
finance_prompt = FINANCE_AGENT_SYSTEM_MESSAGE.format(date=get_today_str())

finance_agent = create_react_agent(
    model=LLM,
    tools=[web_search, think_tool],
    prompt=finance_prompt,
    name="finance_agent"
)

In [ ]:
# for chunk in finance_agent.stream(
#     {"messages": [{"role": "user", "content": "MarathonXP from Kenya"}]}
# ):
#     pretty_print_messages(chunk)

# Reputation Agent

In [ ]:
import sys
sys.path.append("../src")

from prompts.agents import  REPUTATION_AGENT_SYSTEM_MESSAGE

reputation_prompt = REPUTATION_AGENT_SYSTEM_MESSAGE.format(date=get_today_str())

reputation_agent = create_react_agent(
    model=LLM,
    tools=[web_search, think_tool],
    prompt=reputation_prompt,
    name="reputation_agent"
)

# Governance Agent

In [ ]:
import sys
sys.path.append("../src")

from prompts.agents import  GOVERNANCE_AGENT_SYSTEM_MESSAGE

governance_prompt = GOVERNANCE_AGENT_SYSTEM_MESSAGE.format(date=get_today_str())

governance_agent = create_react_agent(
    model=LLM,
    tools=[web_search, think_tool],
    prompt=governance_prompt,
    name="governance_agent"
)

In [ ]:
for chunk in governance_agent.stream(
    {"messages": [{"role": "user", "content": "MarathonXP from Kenya"}]}
):
    pretty_print_messages(chunk)

# Risk Agent

In [ ]:
import sys
sys.path.append("../src")

from prompts.agents import  RISK_AGENT_SYSTEM_MESSAGE

risk_prompt = RISK_AGENT_SYSTEM_MESSAGE.format(date=get_today_str())

risk_agent = create_react_agent(
    model=LLM,
    tools=[web_search, think_tool],
    prompt=risk_prompt,
    name="risk_agent"
)

# Supervisor Agent

In [ ]:
from langgraph_supervisor import create_supervisor
from langchain.chat_models import init_chat_model
from prompts.agents import SUPERVISOR_SYSTEM_MESSAGE

supervisor = create_supervisor(
    model=init_chat_model(LLM, temperature=0),
    agents=[legal_agent, finance_agent, reputation_agent, governance_agent, risk_agent],
    tools=[web_search, think_tool],
    prompt=SUPERVISOR_SYSTEM_MESSAGE,
    add_handoff_back_messages=True,
    output_mode="full_history",
).compile()

In [ ]:
from IPython.display import display, Image

display(Image(supervisor.get_graph().draw_mermaid_png()))

In [ ]:
# for chunk in supervisor.stream(
#     {
#         "messages": [
#             {
#                 "role": "user",
#                 "content": "Do a thorough due diligence on Safaricom from Kenya" ,
#             }
#         ]
#     },
# ):
#     print(chunk)

# final_message_history = chunk["supervisor"]["messages"]

In [ ]:
res = await supervisor.ainvoke({"messages": [{"role": "user", "content": "Do a thorough due diligence on Safaricom from Kenya. That is legal, financial, reputation, governance, and risk aspects."}]})

In [ ]:
for message in res["messages"]:
    print(f"{message.content}\n\n")